In [ ]:
import os
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import benchutils as bu

plt.style.use('bmh')

PYTHON_PATH = '/home/andreas/mambaforge/envs/symdel/bin/python'
WORKER_SCRIPT = 'tmp/symscan_ncpu_worker.py'
N_SEQUENCE = 1_000_000
MAX_DISTANCE = 2
DISTANCE_TYPE = 'levenshtein'
N_REPS = 1

# Sweep 1..N threads over whatever cores this process was pinned to by
# bench_pinned.sh (one logical CPU per physical core, one core class).
# Unpinned this falls back to the whole machine, which is fine for a smoke
# test but not for a scaling curve.
MAX_NCPU = bu.available_cpus()
ncpus = np.arange(1, MAX_NCPU + 1, 1)
print(f'scaling over {MAX_NCPU} cores: {bu.affinity_list()}')

In [2]:
bu.describe_env()

{'colab': False,
 'platform': 'Linux-6.8.0-136-generic-x86_64-with-glibc2.39',
 'python': '3.12.13',
 'git_sha': '04dcf0d',
 'cpu_model': '12th Gen Intel(R) Core(TM) i7-1260P',
 'n_cpus_total': 16,
 'affinity': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
 'n_cpus_visible': 16,
 'governor': 'powersave',
 'no_turbo': '0',
 'gpu': {'name': 'NVIDIA T550 Laptop GPU',
  'memory_total': '4096 MiB',
  'clocks_max_sm': '2100 MHz',
  'clocks_applications_gr': '1065 MHz'},
 'thread_env': {'RAYON_NUM_THREADS': None,
  'OMP_NUM_THREADS': None,
  'MKL_NUM_THREADS': None,
  'OPENBLAS_NUM_THREADS': None,
  'NUMEXPR_NUM_THREADS': None,
  'NUMBA_NUM_THREADS': None,
  'OMP_PROC_BIND': None,
  'OMP_PLACES': None},
 'packages': {'symscan': '0.8.3',
  'pyrepseq': '1.5.2',
  'pybktree': '1.1',
  'rapidfuzz': '3.14.5',
  'pwseqdist': '0.6',
  'numba': '0.66.0',
  'numpy': '2.5.1',
  'scipy': '1.18.0',
  'pandas': '3.0.5'},
 'timeout_seconds': 100}

In [3]:
!mkdir -p tmp

In [4]:
%%writefile tmp/symscan_ncpu_worker.py
import sys
import time
import pandas as pd

import symscan


def main():
    n_sequence, max_distance, distance_type = sys.argv[1:5]
    n_sequence = int(n_sequence)
    max_distance = int(max_distance)

    N_FILES=1
    seqs = []
    for i in range(1,N_FILES+1):
        seqs += pd.read_csv(f'../data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()
    seqs = seqs[:n_sequence]

    t0 = time.perf_counter()
    symscan.get_neighbors_within(seqs, max_distance=max_distance, distance_type=distance_type)
    print(time.perf_counter() - t0)


if __name__ == '__main__':
    main()

Overwriting tmp/symscan_ncpu_worker.py


In [ ]:
def measure_runtime_seconds(n_cpu, n_sequence=N_SEQUENCE, max_distance=MAX_DISTANCE, distance_type=DISTANCE_TYPE):
    cmd = [PYTHON_PATH, WORKER_SCRIPT, str(n_sequence), str(max_distance), distance_type]
    env = os.environ | {'RAYON_NUM_THREADS': str(n_cpu)}
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return float(result.stdout)


In [ ]:
rows = []
for rep in range(N_REPS):
    for n_cpu in ncpus:
        runtime_s = measure_runtime_seconds(n_cpu)
        rows.append({'algorithm': 'symscan', 'n_cpu': int(n_cpu), 'n_sequence': N_SEQUENCE,
                      'distance': MAX_DISTANCE, 'measure': DISTANCE_TYPE,
                      'runtime_s': runtime_s})
        print(n_cpu, rep, runtime_s)

ncpu_df = pd.DataFrame(rows)
ncpu_df.to_csv('../data/symscan_ncpu_benchmark.csv')